<a href="https://colab.research.google.com/github/YuYing33/CS521_FA26_HW/blob/main/multi_norm_adversarial_training_cifar10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [1]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 161MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=b02039f1-3ac4-4c94-88e1-28a161ace616
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 123MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 191MB/s]


In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [3]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [4]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [5]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)

    if rand_init:
        delta = torch.empty_like(x).uniform_(-eps, eps)

    delta = clip_box(x, delta)

    for _ in range(steps):
        grad = input_grad(model, x, delta, y)

        delta = delta + alpha * grad.sign()
        delta = delta.clamp(-eps, eps)
        delta = clip_box(x, delta)

    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)

    if rand_init:
        delta = torch.randn_like(x)
        delta = delta / l2_norm(delta).clamp_min(1e-12)

        radius = torch.rand(
            x.shape[0], 1, 1, 1, device=x.device
        ) ** (1.0 / (x[0].numel()))

        delta = delta * eps * radius

    delta = clip_box(x, delta)

    for _ in range(steps):
        grad = input_grad(model, x, delta, y)

        grad = grad / l2_norm(grad).clamp_min(1e-12)
        delta = delta + alpha * grad

        norm = l2_norm(delta).clamp_min(1e-12)
        delta = delta * (eps / norm).clamp(max=1.0)

        delta = clip_box(x, delta)

    return (x + delta).detach()


In [6]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [7]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [ ]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [8]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [9]:
def single_norm_loss(model, x, y, norm):
    model.eval()

    x_adv = attack(model, x, y, norm)

    model.train()

    loss = F.cross_entropy(model(x_adv), y)

    return loss


In [10]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

epoch 1: loss 1.203 (42 s)
epoch 2: loss 1.181 (83 s)
epoch 3: loss 1.151 (123 s)
epoch 4: loss 1.126 (163 s)
epoch 5: loss 1.097 (204 s)
epoch 6: loss 1.068 (244 s)
epoch 7: loss 1.049 (285 s)
epoch 8: loss 1.030 (325 s)
epoch 9: loss 1.013 (366 s)
epoch 10: loss 1.007 (406 s)
clean 78.6  linf 44.8  l2 56.9  l1 20.3  union 20.2   (121 s)


{'clean': 78.6, 'linf': 44.8, 'l2': 56.9, 'l1': 20.3, 'union': 20.2}

In [12]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

epoch 1: loss 0.892 (41 s)
epoch 2: loss 0.820 (81 s)
epoch 3: loss 0.782 (122 s)
epoch 4: loss 0.748 (162 s)
epoch 5: loss 0.709 (203 s)
epoch 6: loss 0.669 (244 s)
epoch 7: loss 0.625 (284 s)
epoch 8: loss 0.604 (325 s)
epoch 9: loss 0.584 (365 s)
epoch 10: loss 0.577 (406 s)
clean 82.3  linf 32.1  l2 60.3  l1 27.2  union 23.9   (121 s)


{'clean': 82.3, 'linf': 32.1, 'l2': 60.3, 'l1': 27.2, 'union': 23.9}

In [14]:
torch.save(model_l2.state_dict(), "p3_l2_model.pt")
torch.save(model_linf.state_dict(), "p3_linf_model.pt")

In [15]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

epoch 1: loss 1.231 (42 s)
epoch 2: loss 0.828 (83 s)
epoch 3: loss 0.590 (124 s)
epoch 4: loss 0.453 (165 s)
epoch 5: loss 0.337 (206 s)
epoch 6: loss 0.264 (247 s)
epoch 7: loss 0.211 (288 s)
epoch 8: loss 0.175 (329 s)
epoch 9: loss 0.154 (370 s)
epoch 10: loss 0.154 (412 s)
clean 83.7  linf 0.0  l2 0.0  l1 25.8  union 0.0   (121 s)


{'clean': 83.7, 'linf': 0.0, 'l2': 0.0, 'l1': 25.8, 'union': 0.0}

In [16]:
base_model = load_base_model()

print("===== Base Model =====")
base_results = test_model(base_model)

===== Base Model =====
clean 78.7  linf 48.4  l2 60.1  l1 22.4  union 22.3   (122 s)


In [ ]:
def standard_loss(model, x, y):
    model.train()
    return F.cross_entropy(model(x), y)

standard_model = adv_train(
    make_model(),
    standard_loss,
    epochs=5
)


epoch 1: loss 1.884 (8 s)
epoch 2: loss 1.548 (14 s)
epoch 3: loss 1.386 (19 s)
epoch 4: loss 1.244 (25 s)
epoch 5: loss 1.161 (31 s)


In [ ]:
adv_model = adv_train(
    make_model(),
    single_norm_loss,
    norm="linf",
    epochs=5
)


epoch 1: loss 2.202 (49 s)
epoch 2: loss 2.091 (93 s)
epoch 3: loss 2.041 (140 s)
epoch 4: loss 2.020 (186 s)
epoch 5: loss 2.003 (232 s)


In [ ]:
print("===== Standard Model =====")
standard_results = test_model(standard_model)

print("\n===== Adversarial Model =====")
adv_results = test_model(adv_model)


===== Standard Model =====
clean 56.8  linf 0.1  l2 7.5  l1 1.7  union 0.1   (142 s)

===== Adversarial Model =====
clean 36.8  linf 23.6  l2 29.6  l1 20.5  union 19.2   (140 s)


In [ ]:
torch.save(standard_model.state_dict(), "p1_standard_model.pt")
torch.save(adv_model.state_dict(), "p1_adv_model.pt")

print("Both models saved!")


Both models saved!


In [ ]:
def evaluate_linf_eps(model, eps, steps=50, restarts=2):
    model.eval()
    torch.manual_seed(0)

    correct = 0
    total = 0

    for i in range(0, len(x_test), 250):
        x = x_test[i:i+250]
        y = y_test[i:i+250]

        with torch.no_grad():
            robust = model(x).argmax(1) == y

        for _ in range(restarts):
            x_adv = pgd_linf(
                model, x, y,
                eps=eps,
                alpha=2.5 * eps / steps,
                steps=steps
            )

            with torch.no_grad():
                robust &= model(x_adv).argmax(1) == y

        correct += robust.sum().item()
        total += len(y)

    return 100 * correct / total


for eps in [4/255, 16/255]:
    print(f"\nEpsilon = {eps*255:.0f}/255")

    std_acc = evaluate_linf_eps(standard_model, eps)
    adv_acc = evaluate_linf_eps(adv_model, eps)

    print(f"Standard Model: {std_acc:.1f}%")
    print(f"Adversarial Model: {adv_acc:.1f}%")



Epsilon = 4/255
Standard Model: 4.5%
Adversarial Model: 29.8%

Epsilon = 16/255
Standard Model: 0.0%
Adversarial Model: 13.3%


In [ ]:
def fgsm_linf(model, x, y, eps):
    model.eval()

    delta = torch.zeros_like(x)
    grad = input_grad(model, x, delta, y)

    delta = eps * grad.sign()
    delta = clip_box(x, delta)

    return (x + delta).detach()


def evaluate_fgsm(model, eps, batch_size=250):
    model.eval()
    correct = 0

    for i in range(0, len(x_test), batch_size):
        x = x_test[i:i + batch_size]
        y = y_test[i:i + batch_size]

        x_adv = fgsm_linf(model, x, y, eps)

        with torch.no_grad():
            pred = model(x_adv).argmax(1)
            correct += (pred == y).sum().item()

    return 100 * correct / len(y_test)


fgsm_results = {}

for eps in [4/255, 8/255, 16/255]:
    std_acc = evaluate_fgsm(standard_model, eps)
    adv_acc = evaluate_fgsm(adv_model, eps)

    fgsm_results[eps] = (std_acc, adv_acc)

    print(f"\nFGSM epsilon = {eps*255:.0f}/255")
    print(f"Standard Model: {std_acc:.1f}%")
    print(f"Adversarial Model: {adv_acc:.1f}%")



FGSM epsilon = 4/255
Standard Model: 11.2%
Adversarial Model: 29.9%

FGSM epsilon = 8/255
Standard Model: 2.5%
Adversarial Model: 24.2%

FGSM epsilon = 16/255
Standard Model: 0.3%
Adversarial Model: 15.9%


In [11]:
import inspect

for fn in [pgd_linf, pgd_l2, single_norm_loss]:
    print(f"\n===== {fn.__name__} =====")
    print(inspect.getsource(fn))


===== pgd_linf =====
def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)

    if rand_init:
        delta = torch.empty_like(x).uniform_(-eps, eps)

    delta = clip_box(x, delta)

    for _ in range(steps):
        grad = input_grad(model, x, delta, y)

        delta = delta + alpha * grad.sign()
        delta = delta.clamp(-eps, eps)
        delta = clip_box(x, delta)

    return (x + delta).detach()


===== pgd_l2 =====
def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)

    if rand_init:
        delta = torch.randn_like(x)
        delta = delta / l2_norm(delta).clamp_min(1e-12)

        radius = torch.rand(
            x.shape[0], 1, 1, 1, device=x.device
        ) ** (1.0 / (x[0].numel()))

        delta = delta * eps * radius

    delta = clip_box(x, delta)

    for _ in range(steps):
        grad = input_grad(model, x, delta, y)

        grad = grad / l2_norm(grad).clamp_min(1e-12)
        delta 

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [22]:
def ramp_loss(model, x, y, lam=0.5):
    model.eval()

    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")

    # Compute adversarial losses
    model.train()

    logits_q = model(x_q)
    logits_r = model(x_r)

    loss_q = F.cross_entropy(
        logits_q, y, reduction="none"
    )
    loss_r = F.cross_entropy(
        logits_r, y, reduction="none"
    )

    max_loss = torch.maximum(
        loss_q, loss_r
    ).mean()


    correct_mask = logits_q.argmax(dim=1) == y

    if correct_mask.any():
        p_q = F.softmax(
            logits_q[correct_mask], dim=1
        )
        log_p_r = F.log_softmax(
            logits_r[correct_mask], dim=1
        )

        kl_loss = F.kl_div(
            log_p_r,
            p_q,
            reduction="batchmean"
        )
    else:
        kl_loss = logits_q.sum() * 0.0

    return max_loss + lam * kl_loss



def gradient_projection(g_n, g_a):
    gn_flat = g_n.reshape(-1)
    ga_flat = g_a.reshape(-1)

    dot = torch.dot(gn_flat, ga_flat)

    gn_norm = torch.norm(gn_flat)
    ga_norm = torch.norm(ga_flat)

    cosine = dot / (
        gn_norm * ga_norm + 1e-12
    )

    return torch.clamp(cosine, min=0.0) * g_n


In [18]:
g_n = torch.tensor([1.0, 0.0])
g_a = torch.tensor([1.0, 1.0])

result = gradient_projection(g_n, g_a)

print("Gradient projection:", result)
print("Expected:", torch.tensor([0.7071, 0.0]))

assert torch.allclose(
    result,
    torch.tensor([0.7071, 0.0]),
    atol=1e-4
)

print("Gradient projection test: PASS")


Gradient projection: tensor([0.7071, 0.0000])
Expected: tensor([0.7071, 0.0000])
Gradient projection test: PASS


In [19]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)

epoch 1: loss 1.435 (87 s)
epoch 2: loss 1.331 (175 s)
epoch 3: loss 1.304 (261 s)
epoch 4: loss 1.281 (348 s)
epoch 5: loss 1.254 (435 s)
epoch 6: loss 1.227 (522 s)
epoch 7: loss 1.199 (609 s)
epoch 8: loss 1.186 (696 s)
epoch 9: loss 1.171 (783 s)
epoch 10: loss 1.165 (870 s)
clean 79.5  linf 40.9  l2 61.8  l1 44.7  union 39.0   (121 s)


{'clean': 79.5, 'linf': 40.9, 'l2': 61.8, 'l1': 44.7, 'union': 39.0}

In [20]:
torch.save(model_ramp.state_dict(), "p3_ramp_model.pt")

In [23]:
ramp_models = {}
ramp_results = {}

for seed in [0, 1]:
    print(f"\n===== RAMP seed={seed} =====")

    model = adv_train(
        load_base_model(),
        ramp_loss,
        gp_beta=0.5,
        seed=seed
    )

    results = test_model(model)

    ramp_models[seed] = model
    ramp_results[seed] = results

    torch.save(
        model.state_dict(),
        f"p3_ramp_seed{seed}.pt"
    )



===== RAMP seed=0 =====
epoch 1: loss 1.432 (87 s)
epoch 2: loss 1.330 (174 s)
epoch 3: loss 1.303 (261 s)
epoch 4: loss 1.281 (348 s)
epoch 5: loss 1.253 (435 s)
epoch 6: loss 1.227 (522 s)
epoch 7: loss 1.199 (609 s)
epoch 8: loss 1.186 (696 s)
epoch 9: loss 1.171 (783 s)
epoch 10: loss 1.166 (870 s)
clean 79.2  linf 41.1  l2 62.0  l1 44.6  union 38.9   (121 s)

===== RAMP seed=1 =====
epoch 1: loss 1.434 (87 s)
epoch 2: loss 1.332 (174 s)
epoch 3: loss 1.299 (260 s)
epoch 4: loss 1.274 (347 s)
epoch 5: loss 1.248 (434 s)
epoch 6: loss 1.222 (520 s)
epoch 7: loss 1.200 (607 s)
epoch 8: loss 1.186 (694 s)
epoch 9: loss 1.178 (781 s)
epoch 10: loss 1.168 (867 s)
clean 79.0  linf 42.4  l2 62.9  l1 46.8  union 40.7   (121 s)


# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [24]:

def my_loss(model, x, y, tau=0.5, gamma=0.2):
    model.eval()

    x_inf = attack(model, x, y, "linf")
    x_l2 = attack(model, x, y, "l2")
    x_l1 = attack(model, x, y, "l1")

    model.train()

    logits_inf = model(x_inf)
    logits_l2 = model(x_l2)
    logits_l1 = model(x_l1)

    loss_inf = F.cross_entropy(logits_inf, y)
    loss_l2 = F.cross_entropy(logits_l2, y)
    loss_l1 = F.cross_entropy(logits_l1, y)

    losses = torch.stack([
        loss_inf, loss_l2, loss_l1
    ])

    weights = F.softmax(
        losses.detach() / tau, dim=0
    )

    weighted_loss = (weights * losses).sum()

    p_inf = F.softmax(logits_inf, dim=1)
    p_l2 = F.softmax(logits_l2, dim=1)
    p_l1 = F.softmax(logits_l1, dim=1)

    # Detached consensus target
    consensus = (
        p_inf + p_l2 + p_l1
    ).detach() / 3.0

    consistency = (
        F.kl_div(
            F.log_softmax(logits_inf, dim=1),
            consensus,
            reduction="batchmean"
        )
        + F.kl_div(
            F.log_softmax(logits_l2, dim=1),
            consensus,
            reduction="batchmean"
        )
        + F.kl_div(
            F.log_softmax(logits_l1, dim=1),
            consensus,
            reduction="batchmean"
        )
    ) / 3.0

    return weighted_loss + gamma * consistency


In [25]:
# Test my_loss on a small batch
test_model_small = load_base_model()
test_model_small.train()

x = x_train[:8]
y = y_train[:8]

loss = my_loss(test_model_small, x, y)

print("Loss:", loss.item())

# Test backward propagation
test_model_small.zero_grad()
loss.backward()

grad_norm = sum(
    p.grad.norm().item()
    for p in test_model_small.parameters()
    if p.grad is not None
)

print("Gradient norm:", grad_norm)

assert torch.isfinite(loss).item()
assert grad_norm > 0

print("my_loss test: PASS")


Loss: 2.2079479694366455
Gradient norm: 70.34244164824486
my_loss test: PASS


In [26]:
import torch
import random
import numpy as np

wamt_models = {}
wamt_results = {}

for seed in [0, 1]:
    print(f"\n===== WAMT seed={seed} =====")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    model = load_base_model()

    model = adv_train(
        model,
        my_loss,
        seed=seed
    )

    results = test_model(model)

    wamt_models[seed] = model
    wamt_results[seed] = results

    torch.save(
        model.state_dict(),
        f"p3_wamt_seed{seed}.pt"
    )



===== WAMT seed=0 =====
epoch 1: loss 1.251 (122 s)
epoch 2: loss 1.165 (243 s)
epoch 3: loss 1.127 (365 s)
epoch 4: loss 1.095 (486 s)
epoch 5: loss 1.071 (608 s)
epoch 6: loss 1.034 (729 s)
epoch 7: loss 1.014 (851 s)
epoch 8: loss 0.993 (973 s)
epoch 9: loss 0.977 (1094 s)
epoch 10: loss 0.971 (1216 s)
clean 78.6  linf 41.1  l2 62.6  l1 46.5  union 39.0   (121 s)

===== WAMT seed=1 =====
epoch 1: loss 1.253 (122 s)
epoch 2: loss 1.159 (244 s)
epoch 3: loss 1.127 (365 s)
epoch 4: loss 1.092 (487 s)
epoch 5: loss 1.065 (609 s)
epoch 6: loss 1.035 (730 s)
epoch 7: loss 1.009 (852 s)
epoch 8: loss 0.986 (973 s)
epoch 9: loss 0.975 (1095 s)
epoch 10: loss 0.970 (1216 s)
clean 77.7  linf 41.8  l2 62.1  l1 47.2  union 40.2   (121 s)


In [27]:
metrics = ["clean", "linf", "l2", "l1", "union"]

print("===== WAMT Mean Results =====")

for metric in metrics:
    mean_value = (
        wamt_results[0][metric]
        + wamt_results[1][metric]
    ) / 2

    print(f"{metric}: {mean_value:.2f}%")


===== WAMT Mean Results =====
clean: 78.15%
linf: 41.45%
l2: 62.35%
l1: 46.85%
union: 39.60%


In [28]:
def my_loss_ablation(model, x, y):
    return my_loss(model, x, y, tau=0.5, gamma=0.0)

print("===== WAMT Ablation: gamma=0 =====")

ablation_model = adv_train(
    load_base_model(),
    my_loss_ablation,
    epochs=5,
    seed=0
)

ablation_results = test_model(ablation_model)

torch.save(
    ablation_model.state_dict(),
    "p3_wamt_ablation.pt"
)

print("Ablation results:", ablation_results)


===== WAMT Ablation: gamma=0 =====
epoch 1: loss 1.247 (122 s)
epoch 2: loss 1.156 (243 s)
epoch 3: loss 1.110 (365 s)
epoch 4: loss 1.074 (486 s)
epoch 5: loss 1.057 (608 s)
clean 77.6  linf 43.5  l2 62.8  l1 46.4  union 40.6   (120 s)
Ablation results: {'clean': 77.6, 'linf': 43.5, 'l2': 62.8, 'l1': 46.4, 'union': 40.6}


In [29]:
print("===== Strong Attack Evaluation =====")

best_model = wamt_models[1]

strong_results = test_model(
    best_model,
    steps=200,
    restarts=5
)

print("\nDefault Evaluation:")
print(wamt_results[1])

print("\nStrong Evaluation:")
print(strong_results)


===== Strong Attack Evaluation =====
clean 77.7  linf 41.2  l2 62.1  l1 45.4  union 39.4   (1199 s)

Default Evaluation:
{'clean': 77.7, 'linf': 41.8, 'l2': 62.1, 'l1': 47.2, 'union': 40.2}

Strong Evaluation:
{'clean': 77.7, 'linf': 41.2, 'l2': 62.1, 'l1': 45.4, 'union': 39.4}


In [ ]:
print("===== RAMP Equal-Time Comparison =====")

ramp_equal_time = adv_train(
    load_base_model(),
    ramp_loss,
    gp_beta=0.5,
    epochs=14,
    seed=0
)

equal_time_results = test_model(ramp_equal_time)

torch.save(
    ramp_equal_time.state_dict(),
    "p3_ramp_equal_time.pt"
)

print("Equal-time RAMP:", equal_time_results)


===== RAMP Equal-Time Comparison =====
epoch 1: loss 1.431 (87 s)
epoch 2: loss 1.331 (174 s)
epoch 3: loss 1.306 (261 s)
epoch 4: loss 1.286 (348 s)
epoch 5: loss 1.263 (434 s)
epoch 6: loss 1.234 (521 s)
epoch 7: loss 1.206 (608 s)


In [ ]:
# model_mine = adv_train(load_base_model(), my_loss)
# test_model(model_mine)